In [1]:
from pathlib import Path

ROOT = Path.cwd()
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/matthewraymond/bootcamp_matthew_raymond/homework/homework05

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print('RAW ->', RAW.resolve())
print('PROC ->', PROC.resolve())

RAW -> /Users/matthewraymond/bootcamp_matthew_raymond/homework/homework05/data/raw
PROC -> /Users/matthewraymond/bootcamp_matthew_raymond/homework/homework05/data/processed


In [4]:
import numpy as np
dates = pd.date_range('2024-01-01', periods=20, freq='D')
df = pd.DataFrame({'date': dates, 'ticker': ['AAPL']*20, 'price': 150 + np.random.randn(20).cumsum()})
df.head()


,date,ticker,price
0,2024-01-01,AAPL,150.101474
1,2024-01-02,AAPL,149.387455
2,2024-01-03,AAPL,150.531835
3,2024-01-04,AAPL,150.650840
4,2024-01-05,AAPL,152.552610


In [5]:
def ts(): return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

# Save CSV
csv_path = RAW / f"sample_{ts()}.csv"
df.to_csv(csv_path, index=False)
csv_path

PosixPath('data/raw/sample_20260819-181410.csv')

In [6]:
# Save Parquet
pq_path = PROC / f"sample_{ts()}.parquet"
try:
    df.to_parquet(pq_path)
except Exception as e:
    print('Parquet engine not available. Install pyarrow or fastparquet to complete this step.')
    pq_path = None
pq_path

PosixPath('data/processed/sample_20260819-181437.parquet')

In [7]:
def validate_loaded(original, reloaded):
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'date_is_datetime': pd.api.types.is_datetime64_any_dtype(reloaded['date']) if 'date' in reloaded.columns else False,
        'price_is_numeric': pd.api.types.is_numeric_dtype(reloaded['price']) if 'price' in reloaded.columns else False,
    }
    return checks

df_csv = pd.read_csv(csv_path, parse_dates=['date'])
validate_loaded(df, df_csv)

{'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}

In [17]:
try:
    df_pq = pd.read_parquet(pq_path)
    result = validate_loaded(df, df_pq)
    print("Validation result:", result)
except Exception as e:
    print('Parquet read failed:', e)

Validation result: {'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}


In [23]:
import typing as t, pathlib

def detect_format(path: t.Union[str, pathlib.Path]):
    s = str(path).lower()
    if s.endswith('.csv'): return 'csv'
    if s.endswith('.parquet') or s.endswith('.pq') or s.endswith('.parq'): return 'parquet'
    raise ValueError('Unsupported format: ' + s)

def write_df(df: pd.DataFrame, path: t.Union[str, pathlib.Path]):
    p = pathlib.Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    fmt = detect_format(p)
    if fmt == 'csv':
        df.to_csv(p, index=False)
    else:
        try:
            df.to_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e
    return p

def read_df(path: t.Union[str, pathlib.Path]):
    p = pathlib.Path(path)
    fmt = detect_format(p)
    if fmt == 'csv':
        return pd.read_csv(p, parse_dates=['date']) if 'date' in pd.read_csv(p, nrows=0).columns else pd.read_csv(p)
    else:
        try:
            return pd.read_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e


In [26]:
try:
    write_df(df, p_pq)
    result_pq = read_df(p_pq)
    print(result_pq.head())
except RuntimeError as e:
    print('Skipping Parquet util demo:', e)

        date ticker       price
0 2024-01-01   AAPL  150.101474
1 2024-01-02   AAPL  149.387455
2 2024-01-03   AAPL  150.531835
3 2024-01-04   AAPL  150.650840
4 2024-01-05   AAPL  152.552610


In [25]:
write_df(df, p_csv)
result = read_df(p_csv)
print(result.head())

        date ticker       price
0 2024-01-01   AAPL  150.101474
1 2024-01-02   AAPL  149.387455
2 2024-01-03   AAPL  150.531835
3 2024-01-04   AAPL  150.650840
4 2024-01-05   AAPL  152.552610
